# 课后练习解答（04.04_yolo_postprocess_cpu_nms_baseline）

本解答对应《YOLO 后处理与 CPU NMS 基线》课后练习，共 15 题。


### 问题1（单选题）

**题目：** YOLOv5s 输出 `[N,85]` 中前 4 个数通常表示什么？

A. xywh 边框信息
B. 类别名称字符串
C. NPU 内存地址
D. 图片路径

**解答：** A

**解析：** 前 4 维通常是中心点和宽高形式的边框预测。


### 问题2（单选题）

**题目：** CPU NMS baseline 的主要作用是？

A. 作为自定义 NPU NMS 的正确性参考
B. 替代 ATC 转换
C. 删除 OM 模型
D. 安装 CANN

**解答：** A

**解析：** 先有 CPU 参考结果，才能验证自定义算子输出是否一致。


### 问题3（单选题）

**题目：** 后处理中过滤候选框时，常用的分数计算是？

A. objectness * class score
B. x_center + y_center
C. 图片宽度 * 高度
D. NPU 温度 / 功耗

**解答：** A

**解析：** YOLO 常用目标置信度乘类别置信度作为最终 score。


### 问题4（单选题）

**题目：** NMS 的核心目的是？

A. 抑制高度重叠的冗余框
B. 把图片转成 NCHW
C. 把 ONNX 转成 OM
D. 启动 Jupyter

**解答：** A

**解析：** NMS 通过 IoU 阈值去除重复检测框。


### 问题5（多选题）

**题目：** YOLO 后处理通常包含哪些步骤？

A. 坐标从 xywh 转 xyxy
B. 按 score 阈值过滤
C. 按类别或全局执行 NMS
D. 保留最终检测框和类别

**解答：** A、B、C、D

**解析：** 这些步骤共同把模型输出转换为可解释检测结果。


### 问题6（多选题）

**题目：** CPU baseline 中需要保存或观察哪些信息，方便后续 NPU 对齐？

A. boxes
B. scores
C. keep 索引
D. count 数量

**解答：** A、B、C、D

**解析：** 这些信息是自定义 NMS 调用和正确性比较的基础。


### 问题7（多选题）

**题目：** 影响 NMS 输出的参数包括哪些？

A. score_threshold
B. nms_iou_threshold
C. max_detections
D. class_agnostic

**解答：** A、B、C、D

**解析：** 阈值、最大输出数和是否按类别处理都会影响结果。


### 问题8（判断题）

**题目：** CPU NMS baseline 的意义只在于测速，不需要用于正确性对齐。

**解答：** 错误

**解析：** 本实验中 CPU baseline 首先是正确性参考，其次才是性能对比基线。


### 问题9（判断题）

**题目：** 如果输入候选框已经按 score 降序排列，NMS kernel 内部可以暂时不实现排序。

**解答：** 正确

**解析：** 本实验的最小验证版就是先由 Python 侧排序，再让 kernel 专注 IoU 抑制。


### 问题10（填空题）

**题目：** NMS 中判断两个框重叠程度的常用指标是 `____`。

**解答：** IoU

**解析：** IoU 表示交并比，是 NMS 抑制的核心依据。


### 问题11（填空题）

**题目：** 本实验默认的 NMS IoU 阈值是 `____`。

**解答：** 0.45

**解析：** 该值来自 yolo_edge.yaml 的 postprocess.nms_iou_threshold。


### 问题12（简答题）

**题目：** 为什么在开发自定义算子前必须先有 CPU baseline？

**解答：** CPU baseline 提供可理解、可调试的参考结果。自定义算子开发完成后，可以用 keep/count 是否一致来验证功能正确性，避免只看程序是否运行成功。

**解析：** 算子优化不能脱离正确性基准。


### 问题13（简答题）

**题目：** class_agnostic=false 时，NMS 与 class_agnostic=true 有什么区别？

**解答：** class_agnostic=false 会按类别分别做 NMS，不同类别之间互不抑制；true 则把所有候选框放在一起抑制。

**解析：** 是否按类别执行会影响最终检测框数量和保留索引。


### 问题14（简答题）

**题目：** 为什么真实 YOLO 输出过滤后候选框数量可能远大于最初小样例的 51？

**解答：** 小样例通常是为了快速验证自定义算子流程手工准备的少量候选框；真实 OM 输出包含 25200 个预测位置，经过 score 阈值过滤后仍可能剩下上万候选框。

**解析：** 这说明实验五更接近真实 YOLO 后处理场景。


### 问题15（代码设计题）

**题目：** 写一段代码，计算 YOLO 候选框最终 scores 并按阈值过滤。

**解答：**

```python
raw = pred[0].astype(np.float32)  # [25200, 85]
boxes_xywh = raw[:, :4]
obj = raw[:, 4]
cls_scores = raw[:, 5:]
class_ids = cls_scores.argmax(axis=1)
scores = obj * cls_scores[np.arange(raw.shape[0]), class_ids]
mask = scores >= 0.25
boxes_filtered = boxes_xywh[mask]
scores_filtered = scores[mask]
```

**解析：** 后续还需要坐标转换、排序和 NMS。
